### Shortcut preparation of Images based on FLugnummer. Only the images that are not georeferenced will be considered for shortcut preparation. 

In [4]:
import os
import csv
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


def get_user_input():
    start = int(input("Start Flugnummer: "))
    end = int(input("End Flugnummer: "))
    csv_path = input(
        "CSV output path (e.g. C:\\temp\\output.csv or C:\\temp): "
    ).strip()

    return start, end, csv_path


def extract_imagenummer(filename):
    """
    Example:

    1001#00200#US31_2571#ID0821#11_09_1944.tif
    1001#00200#US31_2571#ID0821#11_09_1944_modifiziert.tif

    -> 1001
    """

    try:
        return Path(filename).stem.split("#")[0]
    except Exception:
        return None


def process_folder(folder_path, folder_name, start, end):

    results = []

    image_exts = (
        ".tif",
        ".tiff",
        ".jpg",
        ".jpeg",
        ".png",
    )

    parts = folder_name.split("#")

    if len(parts) != 3:
        return results

    try:
        flugnummer = int(parts[1])
        sortie = parts[2]

    except ValueError:
        return results

    if not (start <= flugnummer <= end):
        return results

    for root, _, files in os.walk(folder_path):

        for file in files:

            if not file.lower().endswith(image_exts):
                continue

            full_path = os.path.join(root, file)

            is_modified = "modi" in file.lower()

            imagenummer = extract_imagenummer(file)

            results.append(
                {
                    "flugnummer": flugnummer,
                    "sortie": sortie,
                    "imagenummer": imagenummer,
                    "filename": file,
                    "path": full_path,
                    "is_modified": is_modified,
                    "needs_shortcut": not is_modified,
                }
            )

    return results


def parallel_scan(path_b, start, end):

    all_results = []

    folders = [
        (f, os.path.join(path_b, f))
        for f in os.listdir(path_b)
        if os.path.isdir(os.path.join(path_b, f))
    ]

    with ThreadPoolExecutor(max_workers=8) as executor:

        futures = [
            executor.submit(
                process_folder,
                folder_path,
                folder_name,
                start,
                end,
            )
            for folder_name, folder_path in folders
        ]

        for future in as_completed(futures):
            all_results.extend(future.result())

    return all_results


def apply_modified_exclusion(results):
    """
    Rule:

    If ANY file with 'modi' exists for a given
    (flugnummer, sortie, imagenummer),

    then ALL records in that group become
    ineligible for shortcut creation.
    """

    groups = defaultdict(list)
    groups_with_modified = set()

    for record in results:

        key = (
            record["flugnummer"],
            record["sortie"],
            record["imagenummer"],
        )

        groups[key].append(record)

        if record["is_modified"]:
            groups_with_modified.add(key)

    for key in groups_with_modified:

        for record in groups[key]:
            record["needs_shortcut"] = False

    return results


def export_csv(results, csv_path, start, end):

    path_obj = Path(csv_path)

    if not path_obj.suffix:
        path_obj = path_obj / f"flugnummer_{start}_{end}.csv"

    path_obj.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:

        with open(
            path_obj,
            mode="w",
            newline="",
            encoding="utf-8",
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "flugnummer",
                    "sortie",
                    "imagenummer",
                    "filename",
                    "path",
                    "is_modified",
                    "needs_shortcut",
                ],
            )

            writer.writeheader()
            writer.writerows(results)

        print(f"\n📄 CSV exported → {path_obj}")

    except PermissionError:

        print(f"\n❌ Permission denied: {path_obj}")
        print("Run as admin or choose another folder.")


def print_stats(results):

    stats = defaultdict(int)
    modified = 0
    shortcut_candidates = 0

    for r in results:

        stats[r["flugnummer"]] += 1

        if r["is_modified"]:
            modified += 1

        if r["needs_shortcut"]:
            shortcut_candidates += 1

    print("\n📊 STATS")
    print("-" * 40)

    print(f"Total images: {len(results)}")
    print(f"Modified images: {modified}")
    print(f"Shortcut candidates: {shortcut_candidates}")

    print()

    for flugnummer in sorted(stats):
        print(
            f"Flugnummer {flugnummer}: "
            f"{stats[flugnummer]}"
        )


def create_shortcut(target, shortcut_path):
    """
    Windows .lnk creation.
    Requires:
        pip install pywin32
    """

    try:
        import win32com.client

        shell = win32com.client.Dispatch("WScript.Shell")

        shortcut = shell.CreateShortCut(shortcut_path)
        shortcut.Targetpath = target
        shortcut.WorkingDirectory = os.path.dirname(target)
        shortcut.save()

        return True

    except Exception as e:
        print(f"❌ Shortcut error: {e}")
        return False


def create_shortcuts(results):

    choice = input(
        "\nCreate shortcuts for eligible images? (y/n): "
    ).strip().lower()

    if choice != "y":
        return

    output_dir = input(
        "Shortcut output directory: "
    ).strip()

    Path(output_dir).mkdir(
        parents=True,
        exist_ok=True,
    )

    eligible = [
        r
        for r in results
        if r["needs_shortcut"]
    ]

    created = 0

    for r in eligible:

        try:

            folder_name = Path(r["path"]).parent.name

            target_folder = os.path.join(
                output_dir,
                folder_name,
            )

            Path(target_folder).mkdir(
                parents=True,
                exist_ok=True,
            )

            shortcut_path = os.path.join(
                target_folder,
                r["filename"] + ".lnk",
            )

            if create_shortcut(
                r["path"],
                shortcut_path,
            ):
                created += 1

        except Exception as e:

            print(
                f"❌ Failed: "
                f"{r['filename']} -> {e}"
            )

    print(
        f"\n📁 Shortcuts created: "
        f"{created}/{len(eligible)}"
    )


def main():

    path_b = r"W:"

    start, end, csv_path = get_user_input()

    print("\n🔍 Scanning (parallel)...")

    results = parallel_scan(
        path_b,
        start,
        end,
    )

    results = apply_modified_exclusion(results)

    print(
        f"✅ Found {len(results)} images"
    )

    export_csv(
        results,
        csv_path,
        start,
        end,
    )

    print_stats(results)

    create_shortcuts(results)


if __name__ == "__main__":
    main()


🔍 Scanning (parallel)...
✅ Found 34 images

📄 CSV exported → flugnummer_200_200.csv

📊 STATS
----------------------------------------
Total images: 34
Modified images: 16
Shortcut candidates: 2

Flugnummer 200: 34

📁 Shortcuts created: 2/2


### Creating shortcut of Images to georeference from 'LBA_Bilder_in_Bestand.xlsx' file

In [2]:
import os
import csv
import re
import sys
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from typing import Dict, List, Tuple, Set, Optional, Any, Union

import pandas as pd

# Try to import win32com for shortcut creation
try:
    import win32com.client
    HAS_WIN32COM = True
except ImportError:
    HAS_WIN32COM = False
    print("⚠️ Warning: pywin32 not installed. Shortcut creation will not work.")
    print("   Install it with: pip install pywin32")


class ImageMatcher:
    """Main class for matching images from Excel catalog to network folders and creating shortcuts."""
    
    def __init__(self):
        """Initialize the ImageMatcher with default values."""
        self.base_paths = [r"W:", r"Y:"]  # Multiple drives to scan
        self.excel_path = None
        self.excel_folder_name = None  # Store the folder name containing the Excel file
        self.csv_output = None
        self.results = []
        self.required_lookup = {}  # Will store (sortie, imagenummer_int, date) -> True
        self.sortie_set = set()
        self.image_extensions = ('.tif', '.tiff', '.jpg', '.jpeg', '.png')
        self.scan_stats = {}  # Track which drive/folder contributed what
        
    def get_user_input(self) -> None:
        """Get user input for Excel directory and CSV output path."""
        print("\n" + "="*60)
        print("IMAGE MATCHER AND SHORTCUT GENERATOR")
        print("="*60)
        print("\n📁 The Excel file must be named: LBA_Bilder_in_Bestand_LBASortie.xlsx")
        
        excel_dir = input("Enter the directory where this file is located (or press Enter for current folder): ").strip()
        if not excel_dir:
            excel_dir = os.getcwd()
        
        # Store the folder name (last part of the path)
        self.excel_folder_name = Path(excel_dir).name
        # self.excel_path = os.path.join(excel_dir, "LBA_Bilder_in_Bestand.xlsx")
        self.excel_path = os.path.join(excel_dir, "LBA_Bilder_in_Bestand_LBASortie.xlsx")
        
        # Validate existence
        if not os.path.isfile(self.excel_path):
            raise FileNotFoundError(f"❌ Excel file not found: {self.excel_path}")
        
        print(f"📂 Excel folder name: {self.excel_folder_name}")
        
        # Ask which drives to scan
        print("\n💾 Which drives should be scanned?")
        print(f"   Default: {', '.join(self.base_paths)}")
        drives_input = input("Enter drive letters (comma-separated, e.g., W,Y or press Enter for default): ").strip()
        
        if drives_input:
            # Parse custom drives
            custom_drives = [d.strip() + ":" for d in drives_input.split(',') if d.strip()]
            if custom_drives:
                self.base_paths = custom_drives
                print(f"📁 Will scan: {', '.join(self.base_paths)}")
        else:
            print(f"📁 Will scan: {', '.join(self.base_paths)}")
        
        # CSV output path - now optional, will generate default if not provided
        csv_output = input("\nCSV output path (press Enter for default location, or provide full path): ").strip()
        
        if not csv_output:
            # Create default output in the same directory as the Excel file
            default_output_dir = Path(excel_dir) / "output"
            default_output_dir.mkdir(parents=True, exist_ok=True)
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            self.csv_output = str(default_output_dir / f"{self.excel_folder_name}_{timestamp}.csv")
            print(f"📁 Using default output: {self.csv_output}")
        else:
            self.csv_output = csv_output
    
    @staticmethod
    def expand_bildnummer(bild_str: str) -> List[int]:
        """
        Expand Bildnummer strings like "23-24 2021–2022" or "1021. 2023".
        
        Handles various formats:
        - Ranges with hyphens or en-dashes: "23-24", "2021–2022"
        - Single numbers: "1021", "2023"
        - Mixed formats: "23-24 2021–2022"
        - Numbers with dots: "1021."
        - Numbers with spaces and line breaks
        
        Args:
            bild_str: String containing image numbers and ranges
            
        Returns:
            List of integers representing all image numbers
        """
        numbers = []
        
        # Replace various separators with spaces
        # Replace en-dash, em-dash with hyphen
        bild_str = bild_str.replace('–', '-').replace('—', '-')
        # Replace dots, commas, semicolons with spaces
        for separator in ['.', ',', ';', '•', '·']:
            bild_str = bild_str.replace(separator, ' ')
        
        # Split by whitespace and process each token
        tokens = bild_str.split()
        
        for token in tokens:
            if not token:
                continue
            
            # Check if it's a range (contains hyphen)
            if '-' in token:
                parts = token.split('-')
                if len(parts) == 2:
                    try:
                        start = int(parts[0].strip())
                        end = int(parts[1].strip())
                        numbers.extend(range(start, end + 1))
                    except ValueError:
                        # Try to handle cases like "2021-2022" where parts might have extra spaces
                        try:
                            start = int(re.search(r'\d+', parts[0]).group())
                            end = int(re.search(r'\d+', parts[1]).group())
                            numbers.extend(range(start, end + 1))
                        except:
                            pass
            else:
                # Single number
                try:
                    # Extract digits from the token (handles cases like "1021.")
                    num = int(re.search(r'\d+', token).group())
                    numbers.append(num)
                except:
                    pass
        
        return sorted(set(numbers))  # Remove duplicates and sort
    
    @staticmethod
    def extract_imagenumber_int(filename: str) -> Optional[int]:
        """
        Extract image number from filename and convert to integer.
        
        Example:
        "1001#00200#US31_2571#ID0821#11_09_1944.tif" -> 1001
        "0034#..." -> 34
        
        Args:
            filename: Image filename
            
        Returns:
            Image number as integer or None if extraction fails
        """
        try:
            # Get the base filename without any extra extensions
            base = Path(filename).stem
            # Remove any modifier suffixes
            for modifier in ['_modified', '_modifiziert', '_modi']:
                if base.endswith(modifier):
                    base = base[:-len(modifier)]
            # Split by # and take first part
            num_str = base.split("#")[0]
            # Convert to integer (removes leading zeros)
            return int(num_str)
        except Exception:
            return None
    
    @staticmethod
    def extract_image_date(filename: str) -> Optional[str]:
        """
        Extract date from filename (e.g., "11_09_1944" or "06_19_40").
        
        Args:
            filename: Image filename
            
        Returns:
            Date string in format "DD_MM_YYYY" or None if not found
        """
        # Get the base filename without extension
        base = Path(filename).stem
        # Remove any modifier suffixes
        for modifier in ['_modified', '_modifiziert', '_modi']:
            if base.endswith(modifier):
                base = base[:-len(modifier)]
        
        # Split by # and get the last part (should be the date)
        parts = base.split("#")
        if len(parts) >= 1:
            candidate = parts[-1]
            
            # Try to match DD_MM_YYYY pattern
            match = re.match(r'(\d{2})_(\d{2})_(\d{4})', candidate)
            if match:
                return candidate
            
            # Try to match DD_MM_YY pattern and convert to DD_MM_YYYY
            match = re.match(r'(\d{2})_(\d{2})_(\d{2})', candidate)
            if match:
                day, month, year_short = match.groups()
                # Convert 2-digit year to 4-digit (assuming 19xx for 40-99, 20xx for 00-39)
                year_int = int(year_short)
                if year_int >= 40:
                    year_full = f"19{year_short}"
                else:
                    year_full = f"20{year_short}"
                return f"{day}_{month}_{year_full}"
        
        return None
    
    @staticmethod
    def is_modified_file(filename: str) -> bool:
        """
        Check if a file is a modified version.
        
        Args:
            filename: Image filename
            
        Returns:
            True if filename contains 'modified', 'modifiziert', or 'modi'
        """
        name_lower = filename.lower()
        return ('modified' in name_lower or 
                'modifiziert' in name_lower or 
                '_modi' in name_lower or 
                name_lower.endswith('.aux'))  # .aux files are auxiliary/modified
    
    def read_excel_file(self) -> None:
        """Read and parse the Excel file, building the required lookup dictionary."""
        print("\n📂 Reading Excel file...")
        df = pd.read_excel(self.excel_path, dtype=str)
        
        # Normalise column names
        df.columns = df.columns.str.strip()
        required_cols = {"Datum", "Sortie/Flugnr.", "Bildnummer"}
        
        if not required_cols.issubset(df.columns):
            raise ValueError(f"Excel must contain columns: {required_cols}")
        
        # Clear previous data
        self.required_lookup = {}
        self.sortie_set = set()
        skipped_sorties = []
        skipped_dates = []
        
        # Group rows by sortie and date to combine bildnummer from multiple rows
        # This handles cases where Bildnummer is spread across multiple lines
        grouped_data = defaultdict(lambda: {"bildnummer_strings": []})
        
        for idx, row in df.iterrows():
            datum_str = str(row["Datum"]).strip()
            if not datum_str or datum_str == "nan":
                print(f"⚠️ Skipping row {idx + 1}: empty Date")
                continue
            
            # Parse date
            try:
                date_obj = datetime.strptime(datum_str, "%d.%m.%Y")
                date_key = date_obj.strftime("%d_%m_%Y")
            except ValueError:
                try:
                    date_obj = datetime.strptime(datum_str, "%d/%m/%Y")
                    date_key = date_obj.strftime("%d_%m_%Y")
                except ValueError:
                    print(f"⚠️ Skipping row {idx + 1}: invalid date: {datum_str}")
                    skipped_dates.append(datum_str)
                    continue
            
            # Get sortie
            sortie = str(row["Sortie/Flugnr."]).strip()
            if not sortie or sortie == "nan" or sortie == "None":
                print(f"⚠️ Skipping row {idx + 1}: empty Sortie (Date: {datum_str})")
                skipped_sorties.append(datum_str)
                continue
            
            self.sortie_set.add(sortie)
            
            # Get bildnummer
            bild_str = str(row["Bildnummer"]).strip()
            if bild_str and bild_str != "nan":
                grouped_data[(sortie, date_key)]["bildnummer_strings"].append(bild_str)
        
        # Process grouped data
        total_image_numbers = 0
        for (sortie, date_key), data in grouped_data.items():
            # Combine all bildnummer strings
            combined_bild_str = ' '.join(data["bildnummer_strings"])
            
            # Expand to individual image numbers
            image_numbers = self.expand_bildnummer(combined_bild_str)
            
            # Add to lookup
            for num in image_numbers:
                self.required_lookup[(sortie, num, date_key)] = True
                total_image_numbers += 1
        
        print(f"✅ Found {len(self.required_lookup)} unique (sortie, imagenummer, date) targets.")
        print(f"📋 Sorties involved: {', '.join(self.sortie_set)}")
        
        if skipped_sorties:
            print(f"⚠️ Skipped {len(skipped_sorties)} rows due to missing Sortie values")
        if skipped_dates:
            print(f"⚠️ Skipped {len(skipped_dates)} rows due to invalid dates")
    
    def find_modified_version_in_folder(self, folder_path: str, sortie: str, imagenummer: int, date: str) -> bool:
        """
        Check if a modified version exists for the exact (sortie, imagenummer, date) combination.
        
        Args:
            folder_path: Path to the folder containing images
            sortie: Sortie identifier
            imagenummer: Image number to check for modified version (as integer)
            date: Date string in format "DD_MM_YYYY"
            
        Returns:
            True if a modified version exists for this exact combination, False otherwise
        """
        for root, _, files in os.walk(folder_path):
            for file in files:
                # Check if it's a modified file
                if not self.is_modified_file(file):
                    continue
                
                # Extract image number from the file
                file_imagenummer = self.extract_imagenumber_int(file)
                if file_imagenummer != imagenummer:
                    continue
                
                # Extract date from the file
                file_date = self.extract_image_date(file)
                if file_date != date:
                    continue
                
                # Found a modified version with matching image number and date
                return True
                    
        return False
    
    def process_folder(self, folder_path: str, folder_name: str, drive: str) -> List[Dict]:
        """
        Process a single folder to find matching images and check for modified versions.
        
        Args:
            folder_path: Full path to the folder
            folder_name: Name of the folder
            drive: The drive letter (e.g., "W:", "Y:")
            
        Returns:
            List of dictionaries containing matching image information
        """
        results = []
        
        # Check folder name format
        parts = folder_name.split("#")
        if len(parts) != 3:
            return results
        
        # Extract flugnummer and sortie
        flugnummer_raw = parts[1]
        folder_sortie = parts[2]
        
        # Try to convert flugnummer to int, but keep as string if fails
        try:
            flugnummer = int(flugnummer_raw)
        except ValueError:
            flugnummer = flugnummer_raw if flugnummer_raw else "N/A"
        
        # Check if this sortie is in our required set
        if folder_sortie not in self.sortie_set:
            return results
        
        # First pass: collect all matching images in this folder
        matching_images = []
        
        for root, _, files in os.walk(folder_path):
            for file in files:
                # Skip if not an image file
                if not any(file.lower().endswith(ext) for ext in self.image_extensions):
                    continue
                
                full_path = os.path.join(root, file)
                is_modified = self.is_modified_file(file)
                imagenummer = self.extract_imagenumber_int(file)
                
                if imagenummer is None:
                    continue
                    
                image_date = self.extract_image_date(file)
                if image_date is None:
                    continue
                
                # Check if this combination is required (using folder_sortie from folder name)
                key = (folder_sortie, imagenummer, image_date)
                if key in self.required_lookup:
                    matching_images.append({
                        "drive": drive,
                        "flugnummer": flugnummer,
                        "flugnummer_raw": flugnummer_raw,
                        "sortie": folder_sortie,
                        "imagenummer": imagenummer,
                        "imagenummer_str": f"{imagenummer:04d}",  # Zero-padded for display
                        "date": image_date,
                        "filename": file,
                        "path": full_path,
                        "is_modified": is_modified,
                        "needs_shortcut": True,  # Initially set to True
                    })
        
        # Second pass: check for modified versions for each exact combination
        # Group by (imagenummer, date) since sortie is constant for this folder
        grouped_by_key = defaultdict(list)
        for img in matching_images:
            key = (img["imagenummer"], img["date"])
            grouped_by_key[key].append(img)
        
        # For each combination, check if ANY modified version exists in this folder
        for (imagenummer, date), images in grouped_by_key.items():
            # Check if there's a modified file for this exact combination
            has_modified = any(img["is_modified"] for img in images)
            
            # Also search the entire folder for modified versions with same (sortie, imagenummer, date)
            if not has_modified:
                has_modified = self.find_modified_version_in_folder(
                    folder_path, folder_sortie, imagenummer, date
                )
            
            # If a modified version exists, NO shortcut should be created for ANY version
            # of this exact (imagenummer, date, sortie) combination
            if has_modified:
                for img in images:
                    img["needs_shortcut"] = False
                    if not img["is_modified"]:
                        img["excluded_reason"] = f"Modified version exists for same image number {imagenummer} and date {date}"
        
        return matching_images
    
    def scan_drive(self, drive: str, required_lookup: Dict, sortie_set: Set) -> List[Dict]:
        """
        Scan a single drive for matching images.
        
        Args:
            drive: Drive letter (e.g., "W:", "Y:")
            required_lookup: Dictionary of required combinations
            sortie_set: Set of sorties to look for
            
        Returns:
            List of matching image records
        """
        print(f"\n🔍 Scanning {drive} (parallel)...")
        
        all_results = []
        
        # Get all folders on this drive
        try:
            folders = [
                (f, os.path.join(drive, f)) 
                for f in os.listdir(drive)
                if os.path.isdir(os.path.join(drive, f))
            ]
        except PermissionError as e:
            print(f"❌ Permission denied accessing {drive}: {e}")
            return []
        except FileNotFoundError:
            print(f"❌ Drive {drive} not found or not accessible")
            return []
        
        print(f"📁 Found {len(folders)} folders on {drive}")
        
        # Process folders in parallel
        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = [
                executor.submit(self.process_folder, folder_path, folder_name, drive)
                for folder_name, folder_path in folders
            ]
            
            completed = 0
            for future in as_completed(futures):
                completed += 1
                if completed % 50 == 0:
                    print(f"   {drive}: Processed {completed}/{len(folders)} folders...")
                all_results.extend(future.result())
        
        print(f"✅ Found {len(all_results)} matching images on {drive}")
        return all_results
    
    def parallel_scan(self) -> None:
        """Scan all drives in parallel to find matching images."""
        print("\n" + "="*60)
        print("🔍 SCANNING DRIVES")
        print("="*60)
        print(f"Drives to scan: {', '.join(self.base_paths)}")
        
        all_results = []
        
        # Scan each drive
        for drive in self.base_paths:
            drive_results = self.scan_drive(drive, self.required_lookup, self.sortie_set)
            all_results.extend(drive_results)
            
            # Store stats
            self.scan_stats[drive] = len(drive_results)
        
        self.results = all_results
        
        # Print summary
        print("\n" + "="*60)
        print("📊 SCAN SUMMARY")
        print("="*60)
        for drive, count in self.scan_stats.items():
            print(f"   {drive}: {count} matching images")
        print(f"   TOTAL: {len(self.results)} matching images")
    
    def apply_modified_exclusion(self) -> None:
        """
        Additional global exclusion check across different folders.
        A modified version in one folder should NOT affect images in another folder,
        because each folder has its own sortie context.
        """
        # Group by (sortie, imagenummer, date) across all results
        groups = defaultdict(list)
        groups_with_modified = set()
        
        for record in self.results:
            key = (record["sortie"], record["imagenummer"], record["date"])
            groups[key].append(record)
            if record["is_modified"]:
                groups_with_modified.add(key)
        
        # If a modified version exists in ANY folder for this exact (sortie, imagenummer, date),
        # mark all versions (in any folder) as ineligible
        for key in groups_with_modified:
            for record in groups[key]:
                if not record["is_modified"]:
                    record["needs_shortcut"] = False
                    record["excluded_reason"] = "Modified version exists in another folder for same sortie/image/date"
        
        excluded_count = sum(1 for r in self.results if not r["needs_shortcut"] and not r["is_modified"])
        print(f"\n📊 Modified exclusion applied: {len(groups_with_modified)} groups with modified versions")
        print(f"   Excluded {excluded_count} non-modified images due to presence of modified versions with same (sortie, image number, date)")
    
    def export_csv(self) -> None:
        """Export results to CSV file with foldername in the filename."""
        path_obj = Path(self.csv_output)
        
        # If the path is a directory, create filename with foldername
        if path_obj.is_dir() or not path_obj.suffix:
            # Ensure directory exists
            path_obj.mkdir(parents=True, exist_ok=True)
            # Create filename with foldername and timestamp
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{self.excel_folder_name}_{timestamp}.csv"
            path_obj = path_obj / filename
        
        # Ensure parent directory exists
        path_obj.parent.mkdir(parents=True, exist_ok=True)
        
        try:
            with open(path_obj, 'w', newline='', encoding='utf-8-sig') as f:  # utf-8-sig for Excel compatibility
                # Check if we have exclusion reasons
                has_exclusion_reason = any("excluded_reason" in r for r in self.results)
                fieldnames = [
                    "drive", "flugnummer", "flugnummer_raw", "sortie", "imagenummer", 
                    "imagenummer_str", "date", "filename", "path", "is_modified", "needs_shortcut"
                ]
                if has_exclusion_reason:
                    fieldnames.append("excluded_reason")
                
                writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter=';')  # Use semicolon for Excel compatibility
                writer.writeheader()
                
                for record in self.results:
                    row = {k: v for k, v in record.items() if k in fieldnames}
                    writer.writerow(row)
            
            print(f"\n📄 CSV exported → {path_obj}")
            print(f"   Folder name used in file: {self.excel_folder_name}")
        except PermissionError:
            print(f"\n❌ Permission denied: {path_obj}")
            print("   Please choose a different output location")
    
    def print_stats(self) -> None:
        """Print statistics about the matching results."""
        print("\n" + "="*60)
        print("📊 STATISTICS")
        print("="*60)
        print(f"Excel folder: {self.excel_folder_name}")
        print(f"Total matched images: {len(self.results)}")
        
        # Stats by drive
        if self.scan_stats:
            print("\n📁 By Drive:")
            for drive, count in self.scan_stats.items():
                print(f"   {drive}: {count} images")
        
        modified = sum(1 for r in self.results if r["is_modified"])
        candidates = sum(1 for r in self.results if r["needs_shortcut"] and not r["is_modified"])
        
        print(f"\nModified images (found): {modified}")
        print(f"Shortcut candidates (non-modified, no modified version with same sortie/image/date): {candidates}")
        
        # Show breakdown by flugnummer
        if self.results:
            print("\n📋 By Flugnummer:")
            by_flug = defaultdict(int)
            for r in self.results:
                by_flug[str(r["flugnummer"])] += 1
            
            for flug in sorted(by_flug, key=lambda x: (str(x).isdigit(), str(x))):
                print(f"   Flugnummer {flug}: {by_flug[flug]} images")
            
            # Show breakdown by sortie
            print("\n📋 By Sortie:")
            by_sortie = defaultdict(int)
            for r in self.results:
                by_sortie[r["sortie"]] += 1
            
            for sortie in sorted(by_sortie):
                print(f"   {sortie}: {by_sortie[sortie]} images")
            
            # Show which images were excluded due to modified versions
            excluded = [r for r in self.results if not r["is_modified"] and not r["needs_shortcut"]]
            if excluded:
                print(f"\n⚠️ {len(excluded)} non-modified images were excluded because modified versions exist with matching (sortie, image number, date):")
                for r in excluded[:10]:  # Show first 10
                    reason = r.get("excluded_reason", "Modified version exists")
                    print(f"   - {r['filename']} (Drive: {r.get('drive', 'Unknown')}, imagenummer: {r['imagenummer']}, date: {r['date']}, sortie: {r['sortie']})")
                    print(f"     → {reason}")
                if len(excluded) > 10:
                    print(f"   ... and {len(excluded) - 10} more")
    
    def create_shortcut(self, target_path: str, shortcut_path: str) -> bool:
        """
        Create a Windows shortcut (.lnk file).
        
        Args:
            target_path: Path to the target file
            shortcut_path: Path where the shortcut should be created
            
        Returns:
            True if successful, False otherwise
        """
        if not HAS_WIN32COM:
            print("❌ Cannot create shortcut: pywin32 not installed")
            return False
        
        try:
            # Ensure target path exists
            if not os.path.exists(target_path):
                print(f"❌ Target does not exist: {target_path}")
                return False
            
            # Create shortcut using Windows API
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(shortcut_path)
            shortcut.Targetpath = target_path
            shortcut.WorkingDirectory = os.path.dirname(target_path)
            shortcut.IconLocation = target_path  # Use the file's own icon
            shortcut.save()
            
            return True
            
        except Exception as e:
            print(f"❌ Shortcut creation error for {os.path.basename(target_path)}: {e}")
            return False
    
    def create_shortcuts(self) -> None:
        """Create shortcuts for all eligible images."""
        eligible = [r for r in self.results if r["needs_shortcut"] and not r["is_modified"]]
        
        if not eligible:
            print("\n⚠️ No eligible images found for shortcut creation.")
            print("   (This happens when all matched images have modified versions with same sortie/image/date)")
            return
        
        print(f"\n📁 Found {len(eligible)} eligible images for shortcut creation")
        
        # Show examples of what will be created
        print("\n📋 Examples of images that will get shortcuts:")
        for r in eligible[:5]:
            drive_info = f" (Drive: {r.get('drive', 'Unknown')})"
            print(f"   - {r['filename']}{drive_info} (Sortie: {r['sortie']}, Image: {r['imagenummer']}, Date: {r['date']})")
        if len(eligible) > 5:
            print(f"   ... and {len(eligible) - 5} more")
        
        choice = input("\nCreate shortcuts for these images? (y/n): ").strip().lower()
        if choice != 'y':
            print("Shortcut creation cancelled.")
            return
        
        # Get output directory for shortcuts
        out_dir = input("Shortcut output directory (press Enter for current folder): ").strip()
        if not out_dir:
            out_dir = os.getcwd()
        
        # Create a subfolder with the Excel folder name for shortcuts
        out_path = Path(out_dir) / f"{self.excel_folder_name}_shortcuts"
        out_path.mkdir(parents=True, exist_ok=True)
        
        print(f"\n🔨 Creating shortcuts in: {out_path}")
        
        created = 0
        failed = 0
        skipped = 0
        
        for i, record in enumerate(eligible, 1):
            try:
                # Create folder structure based on original folder name
                folder_name = Path(record["path"]).parent.name
                target_folder = out_path / folder_name
                target_folder.mkdir(parents=True, exist_ok=True)
                
                # Create shortcut path
                shortcut_filename = Path(record["filename"]).stem + ".lnk"
                shortcut_path = target_folder / shortcut_filename
                
                # Skip if shortcut already exists
                if shortcut_path.exists():
                    print(f"   ⏭️  Skipping existing: {shortcut_path.name}")
                    skipped += 1
                    continue
                
                # Create shortcut
                if self.create_shortcut(record["path"], str(shortcut_path)):
                    created += 1
                    if created % 10 == 0:
                        print(f"   ✅ Created {created}/{len(eligible)} shortcuts...")
                else:
                    failed += 1
                    
            except Exception as e:
                print(f"❌ Failed for {record['filename']}: {e}")
                failed += 1
        
        print(f"\n" + "="*60)
        print(f"📁 SHORTCUT CREATION COMPLETE")
        print("="*60)
        print(f"✅ Successfully created: {created}")
        print(f"❌ Failed: {failed}")
        print(f"⏭️  Skipped (already exist): {skipped}")
        print(f"📂 Location: {out_path}")
        
        if created > 0:
            # Open the output folder
            try:
                os.startfile(str(out_path))
            except Exception:
                pass
    
    def run(self) -> None:
        """Main execution method."""
        try:
            # Get user input
            self.get_user_input()
            
            # Read and parse Excel file
            self.read_excel_file()
            
            if not self.required_lookup:
                print("\n⚠️ No valid entries found in Excel file. Please check the format.")
                return
            
            # Scan all drives
            self.parallel_scan()
            
            if not self.results:
                print("\n⚠️ No matching images found on any drive. Check your Excel file and folder structure.")
                return
            
            # Apply additional exclusion rules (cross-folder)
            self.apply_modified_exclusion()
            
            # Export results to CSV
            self.export_csv()
            
            # Print statistics
            self.print_stats()
            
            # Create shortcuts
            self.create_shortcuts()
            
        except FileNotFoundError as e:
            print(f"\n❌ {e}")
        except ValueError as e:
            print(f"\n❌ {e}")
        except PermissionError as e:
            print(f"\n❌ Permission error: {e}")
            print("   Try running as administrator or using a different output location.")
        except Exception as e:
            print(f"\n❌ Unexpected error: {e}")
            import traceback
            traceback.print_exc()


def main():
    """Entry point for the script."""
    matcher = ImageMatcher()
    matcher.run()


if __name__ == "__main__":
    main()


IMAGE MATCHER AND SHORTCUT GENERATOR

📁 The Excel file must be named: LBA_Bilder_in_Bestand_LBASortie.xlsx
📂 Excel folder name: BE2026-1647 Stuttgart Vaihinger Landstraße Straßen- und Kanalarbeiten

💾 Which drives should be scanned?
   Default: W:, Y:
📁 Will scan: W:, Y:

📂 Reading Excel file...
✅ Found 43 unique (sortie, imagenummer, date) targets.
📋 Sorties involved: US7_0208_A, US7_4014, 366_BS_3089_21, 3G_TUD_S_0198, US31_5335, US7_0168_A, US34_3594, 106W_0109, E_0365, US32_0797, US7_2394, US34_3787, US34_3794, 106G_1657, US34_3796, A_0725, US7_2353, 106G_2812, US23_0793, 106G_2780, US7_0207_A, US39_3904

🔍 SCANNING DRIVES
Drives to scan: W:, Y:

🔍 Scanning W: (parallel)...
📁 Found 3288 folders on W:
   W:: Processed 50/3288 folders...
   W:: Processed 100/3288 folders...
   W:: Processed 150/3288 folders...
   W:: Processed 200/3288 folders...
   W:: Processed 250/3288 folders...
   W:: Processed 300/3288 folders...
   W:: Processed 350/3288 folders...
   W:: Processed 400/3288 f